# Spotify Tracks - Exploratory Data Analysis & Preprocessing

This notebook covers the Exploratory Data Analysis (EDA) and Preprocessing steps for the **Spotify Tracks Dataset**. The goal is to clean the dataset, analyze features, and prepare a balanced sample for clustering.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load the Dataset

We load the `spotify_dataset.csv` dataset, which contains audio features and metadata for Spotify tracks.

In [ ]:
csv_path = "../spotify_dataset.csv"
if not os.path.exists(csv_path):
    csv_path = "spotify_dataset.csv"  # fallback if running from root

df = pd.read_csv(csv_path)
print(f"Original dataset shape: {df.shape}")
df.head()

## 2. Data Cleaning & Deduplication

We drop missing values and duplicate records based on `track_id` to prevent redundant items from skewing our clustering.

In [ ]:
# Drop rows with missing ID or Genre
df = df.dropna(subset=['track_id', 'track_genre'])

# Drop duplicate tracks
df = df.drop_duplicates(subset=['track_id'])
print(f"Cleaned dataset shape: {df.shape}")

## 3. Exploratory Data Analysis (EDA)

Let's check the distribution of some of the key audio characteristics (e.g., `danceability`, `energy`, `valence`, `tempo`).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df['danceability'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Danceability Distribution')

sns.histplot(df['energy'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Energy Distribution')

sns.histplot(df['valence'], kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Valence Distribution (Happiness)')

sns.histplot(df['tempo'], kde=True, ax=axes[1, 1], color='violet')
axes[1, 1].set_title('Tempo (BPM) Distribution')

plt.tight_layout()
plt.show()

### Feature Correlations

Let's look at how these audio characteristics correlate with each other.

In [ ]:
features = [
    'popularity', 'duration_ms', 'danceability', 'energy', 'key', 
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 
    'liveness', 'valence', 'tempo', 'time_signature', 'explicit'
]
df['explicit'] = df['explicit'].astype(float)

corr = df[features].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Audio Features Correlation Matrix')
plt.show()

## 4. Stratified Downsampling

The full dataset has ~90k clean records. Running hierarchical clustering, OPTICS, and MDS on this volume is extremely slow and memory intensive. We downsample to ~3,000 tracks while keeping the representation of all 125 genres balanced (stratified sampling by `track_genre`).

In [ ]:
sample_size = 3000
genres = df['track_genre'].unique()
samples_per_genre = max(1, int(sample_size / len(genres)))
print(f"Genres: {len(genres)}, target samples per genre: {samples_per_genre}")

sampled_indices = []
for g, group in df.groupby('track_genre'):
    sampled_indices.extend(group.sample(min(len(group), samples_per_genre), random_state=42).index)

df_sampled = df.loc[sampled_indices].copy()
df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Stratified sample shape: {df_sampled.shape}")
df_sampled['track_genre'].value_counts().head(10)

## 5. Standard Scaling

We standardize the selected 15 numeric features using `StandardScaler` to prepare them for clustering.

In [ ]:
df_sampled['explicit'] = df_sampled['explicit'].astype(float)
X = df_sampled[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Preprocessed and scaled feature matrix shape:", X_scaled.shape)

## 6. Save Sample for Clustering

We save this cleaned, balanced sample to a CSV file so that we can load it directly in the clustering notebook.

In [ ]:
os.makedirs('output', exist_ok=True)
df_sampled.to_csv('output/spotify_sampled_preprocessed.csv', index=False)
print("Saved preprocessed sample to output/spotify_sampled_preprocessed.csv")